# Day 180 — Month 10 Capstone: ReviewPulse Intelligence Platform
**Month 10 | LangChain + MLflow + Evidently + Prompt Engineering + Output Parsers**

| | |
|---|---|
| Dataset | ReviewPulse India — 600 rows, seed=155 |
| Split | Reference: rows 0–399 \| Current: rows 400–599 |
| LLM | Groq free API — llama-3.1-8b-instant |
| **Total Points** | **100 pts + 10★ bonus** |

---

## Business Scenario

You are a freelance AI consultant hired by **ReviewPulse Analytics**, a platform that monitors freelancer marketplace reviews. The client wants a unified intelligence system that:

1. **Classifies** whether a freelancer will be hired again (ML model, tracked in MLflow)
2. **Extracts** structured risk signals from negative reviews (LangChain + Output Parsers)
3. **Monitors** whether review patterns are drifting over time (Evidently)
4. **Demonstrates** that few-shot prompting improves sentiment classification (Prompt Engineering)
5. **Delivers** an integrated NRA executive report combining all signals

Each task stands alone — you can complete them in any order. All tasks use ReviewPulse India (seed=155).

---

## ⚠️ Rules
- **DO NOT MODIFY** the Raw Data cell
- All NRA Numbers must be read from **printed cell output** — never typed from memory
- `savefig()` before `plt.show()` where applicable
- Groq API key from Colab secrets only
- Pin LangChain versions as shown in the install cell

---
## Cell 1 — Install & Imports (run first, then Runtime → Restart)

In [1]:
# ── Step 1: LangChain + MLflow stack (no explicit pydantic pin) ───────────────
!pip install -q \
    langchain==0.2.16 \
    langchain-community==0.2.16 \
    langchain-groq==0.1.9 \
    mlflow==2.13.2 \
    faiss-cpu \
    scikit-learn \
    pandas numpy

print("✅ Step 1 complete — Runtime → Restart Session, then run Cell 1b")

✅ Step 1 complete — Runtime → Restart Session, then run Cell 1b


In [2]:
# ── Step 2: Evidently installed separately to avoid pydantic conflict ─────────
!pip install -q evidently==0.4.16 --no-deps
!pip install -q plotly tqdm

print("✅ Step 2 complete — now run all remaining cells in order")

✅ Step 2 complete — now run all remaining cells in order


In [3]:
# ── Core imports ──────────────────────────────────────────────────────────────
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd

# Sklearn
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.preprocessing import LabelEncoder

# MLflow
import mlflow
import mlflow.sklearn

# Evidently
from evidently.report import Report
from evidently.metric_preset import DataDriftPreset
from evidently.metrics import ColumnDriftMetric

# LangChain
from langchain_groq import ChatGroq
from langchain.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser
from langchain.output_parsers import PydanticOutputParser
from langchain.schema import Document
from langchain_core.pydantic_v1 import BaseModel, Field
from typing import Literal

# Groq API key
from google.colab import userdata
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

print("✅ Imports complete")

✅ Imports complete


---
## ⛔ Cell 2 — RAW DATA (DO NOT MODIFY)

In [4]:
# ══════════════════════════════════════════════════════════════════
# RAW DATA — DO NOT MODIFY ANY LINE IN THIS CELL
# ReviewPulse India | 600 rows | seed=155
# ══════════════════════════════════════════════════════════════════
np.random.seed(155)
N = 600

TEMPLATES = {
    'positive': [
        "Excellent work, delivered on time and exceeded expectations.",
        "Very professional and responsive throughout the project.",
        "Great quality output, would highly recommend this freelancer.",
        "Outstanding communication and technical skills demonstrated.",
        "Completed the task efficiently and with high accuracy.",
    ],
    'negative': [
        "Disappointed with the quality, did not meet specifications.",
        "Poor communication and missed several deadlines during project.",
        "Work was substandard and required significant rework.",
        "Not satisfied with the final deliverable at all.",
        "Failed to understand requirements and delivered incorrect output.",
    ],
    'neutral': [
        "Work was acceptable but there is room for improvement.",
        "Some delays but the final output was satisfactory.",
        "Average quality, met basic requirements nothing more.",
        "Decent effort but communication could be improved.",
        "Results were okay, not exceptional but not bad either.",
    ]
}

sentiments = np.random.choice(['positive','negative','neutral'], N, p=[0.35, 0.33, 0.32])
ratings, reviews = [], []
for s in sentiments:
    if s == 'positive':
        ratings.append(round(np.random.uniform(3.5, 5.0), 1))
    elif s == 'negative':
        ratings.append(round(np.random.uniform(1.0, 2.5), 1))
    else:
        ratings.append(round(np.random.uniform(2.5, 3.5), 1))
    reviews.append(np.random.choice(TEMPLATES[s]))

ratings = np.array(ratings)
hired_again = np.where(
    ratings >= 3.5,
    np.random.choice(['Yes','No'], N, p=[0.85, 0.15]),
    np.random.choice(['Yes','No'], N, p=[0.15, 0.85])
)

df_raw = pd.DataFrame({
    'review_id': range(400, 400 + N),
    'review_text': reviews,
    'rating': ratings,
    'sentiment': sentiments,
    'hired_again': hired_again
})

print(f"Dataset shape: {df_raw.shape}")
print(f"Columns: {list(df_raw.columns)}")
print(f"\nFirst 3 rows:")
print(df_raw.head(3).to_string())

Dataset shape: (600, 5)
Columns: ['review_id', 'review_text', 'rating', 'sentiment', 'hired_again']

First 3 rows:
   review_id                                             review_text  rating sentiment hired_again
0        400   Work was substandard and required significant rework.     1.5  negative          No
1        401  Work was acceptable but there is room for improvement.     3.2   neutral          No
2        402  Completed the task efficiently and with high accuracy.     4.4  positive         Yes


---
## Cell 3 — Split Reference / Current

In [5]:
# ── Cell 3 — Split Reference / Current ────────────────────────────────────────
# Goal: Split the raw dataset into reference (first 400 rows) and current (last 200 rows)
# Method: Use iloc to slice, then print shapes and average ratings.

# Split
df_ref = df_raw.iloc[:400].reset_index(drop=True)
df_cur = df_raw.iloc[400:].reset_index(drop=True)

print(f"Reference shape: {df_ref.shape}")
print(f"Current shape:   {df_cur.shape}")
print(f"Reference avg rating: {df_ref['rating'].mean():.2f}")
print(f"Current avg rating:   {df_cur['rating'].mean():.2f}")

Reference shape: (400, 5)
Current shape:   (200, 5)
Reference avg rating: 3.01
Current avg rating:   2.98


---
# TASK 1 — MLflow Experiment Tracking (20 pts)

**Goal:** Train a LogisticRegression classifier to predict `hired_again` (Yes/No) from `rating`.
Log **3 runs** to MLflow with different regularisation strengths (C=0.1, C=1.0, C=10.0).
Register the best model in the MLflow Model Registry.

| Sub-task | Points |
|----------|--------|
| T1a — Encode target, train/test split (stratify=y, test_size=0.2, random_state=155) | 4 |
| T1b — Log 3 MLflow runs with accuracy, f1_score, roc_auc, and C as params | 8 |
| T1c — Identify best run by ROC-AUC and register model as `HiredAgainClassifier` | 5 |
| T1d — NRA insight (1 bullet, number from cell output) | 3 |

**Required logged metrics for each run:** `accuracy`, `f1_score`, `roc_auc`  
**Required logged param:** `C`

In [6]:
# ── T1a — Encode target & split ──────────────────────────────────────────────
# Goal: Encode hired_again (Yes=1, No=0) and split reference data.
# Method: Use LabelEncoder, then train_test_split with stratify.

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Encode target
le = LabelEncoder()
df_ref['hired_again_enc'] = le.fit_transform(df_ref['hired_again'])  # Yes->1, No->0

X = df_ref[['rating']]
y = df_ref['hired_again_enc']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=155, stratify=y
)

print(f"Train size: {len(X_train)}")
print(f"Test size:  {len(X_test)}")
print("Class distribution in y_train:")
print(y_train.value_counts())

Train size: 320
Test size:  80
Class distribution in y_train:
hired_again_enc
0    190
1    130
Name: count, dtype: int64


In [7]:
# ── T1b — Log 3 MLflow runs ──────────────────────────────────────────────────
# Goal: Train LogisticRegression with C values 0.1, 1.0, 10.0 and log metrics.
# Method: Loop over C values, train model, compute metrics, log to MLflow.

import mlflow
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

mlflow.set_experiment("ReviewPulse_HiredAgain_Classifier")

results = []

for C in [0.1, 1.0, 10.0]:
    with mlflow.start_run():
        # Train model
        model = LogisticRegression(C=C, max_iter=1000, random_state=155)
        model.fit(X_train, y_train)

        # Predictions
        y_pred = model.predict(X_test)
        y_proba = model.predict_proba(X_test)[:, 1]

        # Metrics
        acc = accuracy_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred, average='binary')
        roc = roc_auc_score(y_test, y_proba)

        # Log
        mlflow.log_param("C", C)
        mlflow.log_metric("accuracy", acc)
        mlflow.log_metric("f1_score", f1)
        mlflow.log_metric("roc_auc", roc)
        mlflow.sklearn.log_model(model, "model")

        # Store for summary
        results.append({"C": C, "accuracy": acc, "f1_score": f1, "roc_auc": roc})
        print(f"C={C} → acc={acc:.3f}, f1={f1:.3f}, roc_auc={roc:.3f}")

# Summary table
print("\nSummary of all runs:")
print(pd.DataFrame(results))

C=0.1 → acc=0.825, f1=0.774, roc_auc=0.833
C=1.0 → acc=0.825, f1=0.774, roc_auc=0.833
C=10.0 → acc=0.825, f1=0.774, roc_auc=0.833

Summary of all runs:
      C  accuracy  f1_score   roc_auc
0   0.1     0.825  0.774194  0.832689
1   1.0     0.825  0.774194  0.832689
2  10.0     0.825  0.774194  0.832689


In [8]:
# ── T1c — Register best model ─────────────────────────────────────────────────
# Goal: Identify the best run by ROC-AUC and register it in Model Registry.
# Method: Query MLflow runs, find max roc_auc, then mlflow.register_model.

from mlflow.tracking import MlflowClient

client = MlflowClient()

# Get all runs for the experiment
experiment = mlflow.get_experiment_by_name("ReviewPulse_HiredAgain_Classifier")
runs = mlflow.search_runs(experiment_ids=[experiment.experiment_id])

# Find best run
best_run = runs.loc[runs['metrics.roc_auc'].idxmax()]
best_run_id = best_run['run_id']
best_C = best_run['params.C']
best_acc = best_run['metrics.accuracy']
best_roc = best_run['metrics.roc_auc']

print(f"Best run ID: {best_run_id}")
print(f"Best C: {best_C}")
print(f"Best accuracy: {best_acc:.3f}")
print(f"Best ROC-AUC: {best_roc:.3f}")

# Register the model
model_uri = f"runs:/{best_run_id}/model"
model_version = mlflow.register_model(model_uri, "HiredAgainClassifier")

print(f"Model registered as version {model_version.version}")
print(f"Model URI: {model_uri}")

Best run ID: 4443d71a4b6f4e85a986d972de5410fe
Best C: 10.0
Best accuracy: 0.825
Best ROC-AUC: 0.833
Model registered as version 2
Model URI: runs:/4443d71a4b6f4e85a986d972de5410fe/model


Registered model 'HiredAgainClassifier' already exists. Creating a new version of this model...
Created version '2' of model 'HiredAgainClassifier'.


### T1d — NRA Insight

**Title:** *Classifier Performance on Reference Data*

- **Number:** Best ROC‑AUC = 0.833
- **Reason:** The rating alone explains most of the variance in re‑hire decisions, with a clear threshold at ~3.5; the classifier achieves strong discrimination (ROC‑AUC > 0.8) on the reference sample.
- **Action:** We will deploy the registered model (`HiredAgainClassifier`, version 1) as an API endpoint with a minimum confidence threshold of 0.85, and automatically flag any freelancer with a predicted re‑hire probability below 0.3 for manual review.

---
# TASK 2 — LangChain + PydanticOutputParser (20 pts)

**Goal:** Load the **current** negative reviews (rating < 2.0) as LangChain Documents.
Use an LCEL chain with PydanticOutputParser to extract a `RiskAssessment` object from each review.

| Sub-task | Points |
|----------|--------|
| T2a — Filter current negatives (rating < 2.0), convert to Documents | 4 |
| T2b — Define RiskAssessment Pydantic model, build LCEL chain with PydanticOutputParser | 8 |
| T2c — Run chain on all low-rating reviews, count urgency=True; NRA insight | 8 |

**RiskAssessment schema:**
```python
class RiskAssessment(BaseModel):
    risk_level: Literal['low', 'medium', 'high']
    root_cause: str = Field(description="One-sentence root cause of the issue")
    action: str = Field(description="Recommended platform action")
    urgency: bool = Field(description="True if immediate action needed")
```

In [9]:
# ── T2a — Filter & load as Documents ──────────────────────────────────────────
# Goal: Extract reviews with rating < 2.0 from current split and convert to LangChain Documents.
# Method: Filter df_cur, create Document objects with page_content and metadata.

from langchain.schema import Document

# Filter low-rating reviews
low_rating_df = df_cur[df_cur['rating'] < 2.0]

# Create Documents
docs = []
for idx, row in low_rating_df.iterrows():
    doc = Document(
        page_content=row['review_text'],
        metadata={
            "review_id": int(row['review_id']),
            "rating": float(row['rating']),
            "sentiment": row['sentiment']
        }
    )
    docs.append(doc)

print(f"Number of low-rating Documents: {len(docs)}")

Number of low-rating Documents: 45


In [10]:
# ── T2b — Define RiskAssessment, build LCEL chain ─────────────────────────────
# Goal: Create Pydantic model for risk assessment and an LCEL chain with parser.
# Method: Define BaseModel, instantiate PydanticOutputParser, build ChatPromptTemplate.

from pydantic import BaseModel, Field
from typing import Literal
from langchain_core.output_parsers import PydanticOutputParser
from langchain.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq

# 1. Define RiskAssessment model
class RiskAssessment(BaseModel):
    risk_level: Literal['low', 'medium', 'high']
    root_cause: str = Field(description="One-sentence root cause of the issue")
    action: str = Field(description="Recommended platform action")
    urgency: bool = Field(description="True if immediate action needed")

# 2. Create parser
parser = PydanticOutputParser(pydantic_object=RiskAssessment)

# 3. Build prompt
system_msg = (
    "You are a risk analyst for a freelancer marketplace. "
    "Extract a risk assessment from the review. "
    "Output only a valid JSON object matching the schema. "
    "Do not include any extra text or code."
)
human_msg = (
    "Review: {review_text}\n"
    "Rating: {rating}\n"
    "{format_instructions}"
)
prompt = ChatPromptTemplate.from_messages([
    ("system", system_msg),
    ("human", human_msg)
])

# 4. Instantiate LLM and chain
llm_risk = ChatGroq(model="llama-3.1-8b-instant", temperature=0)
chain = prompt | llm_risk | parser

print("LCEL chain built successfully.")

LCEL chain built successfully.


In [11]:
# ── T2c — Run chain and collect results ──────────────────────────────────────
# Goal: Invoke the chain for each low-rating review and count urgent cases.
# Method: Loop over docs, call chain.invoke, print each result, count urgency=True.

total_urgent = 0
results_risk = []

for doc in docs:
    result = chain.invoke({
        "review_text": doc.page_content,
        "rating": doc.metadata['rating'],
        "format_instructions": parser.get_format_instructions()
    })
    results_risk.append({
        "review_id": doc.metadata['review_id'],
        "risk_level": result.risk_level,
        "urgency": result.urgency,
        "action": result.action
    })
    print(f"Review {doc.metadata['review_id']}: risk={result.risk_level}, urgency={result.urgency}")
    if result.urgency:
        total_urgent += 1

print(f"\nTotal urgent cases: {total_urgent}")

Review 801: risk=high, urgency=True
Review 803: risk=high, urgency=True
Review 818: risk=high, urgency=True
Review 828: risk=high, urgency=True
Review 832: risk=high, urgency=True
Review 834: risk=high, urgency=True
Review 835: risk=high, urgency=True
Review 838: risk=high, urgency=True
Review 839: risk=high, urgency=True
Review 842: risk=high, urgency=True
Review 843: risk=high, urgency=True
Review 844: risk=high, urgency=True
Review 845: risk=high, urgency=True
Review 849: risk=high, urgency=True
Review 856: risk=high, urgency=True
Review 858: risk=high, urgency=True
Review 865: risk=high, urgency=True
Review 874: risk=high, urgency=True
Review 876: risk=high, urgency=True
Review 882: risk=high, urgency=True
Review 893: risk=high, urgency=True
Review 900: risk=high, urgency=True
Review 901: risk=high, urgency=True
Review 902: risk=high, urgency=True
Review 904: risk=high, urgency=True
Review 910: risk=high, urgency=True
Review 914: risk=high, urgency=True
Review 917: risk=high, urgen

### T2 NRA Insight

**Title:** *High-Urgency Risk Reviews in Current Window*

- **Number:** 45 urgent cases detected (all reviews with rating < 2.0).
- **Reason:** Every review below 2.0 contains explicit complaints about missed deadlines, poor quality, or communication failure, triggering the LLM to classify them as urgent – indicating a systemic failure among low‑rated freelancers.
- **Action:** We will set up a real‑time alert that sends a notification to the support team within 5 minutes of any new low‑rating review (rating < 2.0), and automatically assign a senior case manager to contact the client within 1 hour.

---
# TASK 3 — Evidently Drift Monitoring (20 pts)

**Goal:** Run a drift report comparing reference vs current on two columns: `rating` and `hired_again`.
Then run a full DataDriftPreset on all numeric + categorical columns.

| Sub-task | Points |
|----------|--------|
| T3a — Prepare ref/cur DataFrames with correct column types for Evidently | 4 |
| T3b — ColumnDriftMetric for `rating` and `hired_again`; print drift flag + p-value for each | 8 |
| T3c — DataDriftPreset report; print share_of_drifted_columns; NRA insight | 8 |

**Required columns for Evidently report:** `rating`, `hired_again`  
**Note:** Convert `hired_again` to int (1/0) before passing to Evidently.

In [12]:
# ── T3a — Prepare DataFrames for Evidently ────────────────────────────────────
# Goal: Create reference and current DataFrames with correct column types.
# Method: Convert hired_again to int (Yes=1, No=0) and keep rating.

ref_ev = df_ref[['rating', 'hired_again']].copy()
cur_ev = df_cur[['rating', 'hired_again']].copy()

# Encode hired_again
ref_ev['hired_again'] = ref_ev['hired_again'].map({'Yes': 1, 'No': 0})
cur_ev['hired_again'] = cur_ev['hired_again'].map({'Yes': 1, 'No': 0})

print("Reference dtypes:")
print(ref_ev.dtypes)
print(f"Reference shape: {ref_ev.shape}")
print(f"Current shape:   {cur_ev.shape}")

Reference dtypes:
rating         float64
hired_again      int64
dtype: object
Reference shape: (400, 2)
Current shape:   (200, 2)


In [13]:
# ── T3b — ColumnDriftMetric for rating and hired_again ──────────────────────
# Goal: Compute drift for each column individually.
# Method: Build Report with two ColumnDriftMetric, run, extract results.

from evidently.report import Report
from evidently.metrics import ColumnDriftMetric  # <-- THIS WAS MISSING

report_col = Report(metrics=[
    ColumnDriftMetric(column_name='rating'),
    ColumnDriftMetric(column_name='hired_again')
])
report_col.run(reference_data=ref_ev, current_data=cur_ev)

# Extract results
metrics_dict = report_col.as_dict()['metrics']
for metric in metrics_dict:
    col = metric['result']['column_name']
    drift = metric['result']['drift_detected']
    stattest = metric['result']['stattest_name']
    p_val = metric['result'].get('p_value', None)   # Fixed syntax
    if p_val is not None:
        print(f"{col}: drift_detected={drift}, test={stattest}, p-value={p_val:.4f}")
    else:
        print(f"{col}: drift_detected={drift}, test={stattest}, p-value not available")

rating: drift_detected=False, test=K-S p_value, p-value not available
hired_again: drift_detected=False, test=Z-test p_value, p-value not available


In [14]:
# ── T3c — DataDriftPreset ─────────────────────────────────────────────────────
# Goal: Run full drift report and extract share_of_drifted_columns.
# Method: Use DataDriftPreset, extract from as_dict.

from evidently.metric_preset import DataDriftPreset

report_drift = Report(metrics=[DataDriftPreset()])
report_drift.run(reference_data=ref_ev, current_data=cur_ev)

# Extract share_of_drifted_columns
drift_result = report_drift.as_dict()['metrics'][0]['result']
share = drift_result['share_of_drifted_columns']

print(f"Share of drifted columns: {share:.2%}")

Share of drifted columns: 0.00%


### T3 NRA Insight

**Title:** *Data Drift Status — Reference vs Current*

- **Number:** Share of drifted columns = 0.00%.
- **Reason:** The rating and re‑hire distributions are stable between the reference and current windows; no significant drift has emerged yet, suggesting the current production environment remains consistent with the training data.
- **Action:** We will schedule a weekly Evidently report and set an automated retraining trigger if `share_of_drifted_columns` exceeds 0.4 for two consecutive weeks. Until then, the current model remains valid without retraining.

---
# TASK 4 — Prompt Engineering: Zero-Shot vs Few-Shot (20 pts)

**Goal:** Compare zero-shot and few-shot prompting for sentiment classification on 5 reviews from `df_cur`.

| Sub-task | Points |
|----------|--------|
| T4a — Sample 5 reviews (random_state=155), run zero-shot, record predictions vs ground truth | 7 |
| T4b — Run few-shot (3 examples in prompt) on same 5 reviews, record predictions | 7 |
| T4c — Print comparison table (review_id, ground_truth, zero_shot, few_shot); compute zero-shot accuracy, few-shot accuracy; NRA insight | 6 |

**Sample reviews (random_state=155 from df_cur):**
```
review_id=999, sentiment=positive, rating=4.6
review_id=969, sentiment=negative, rating=1.5
review_id=877, sentiment=positive, rating=3.9
review_id=930, sentiment=positive, rating=3.8
review_id=966, sentiment=neutral,  rating=2.8
```

**Zero-shot prompt structure:**
```
Classify the sentiment of this review as exactly one of: positive, negative, neutral.
Review: {review_text}
Sentiment:
```

**Few-shot examples to inject:**
```
Example 1: "Excellent work, delivered on time" → positive
Example 2: "Disappointed with quality, did not meet specs" → negative
Example 3: "Average quality, met basic requirements" → neutral
```

In [15]:
# ── T4a — Sample & zero-shot ──────────────────────────────────────────────────
# Goal: Sample 5 reviews (random_state=155) and get zero-shot predictions.
# Method: Use ChatGroq with a simple zero-shot prompt.
# Improvement: Strip trailing punctuation from prediction to avoid mismatch (e.g., "neutral." -> "neutral").

from langchain_groq import ChatGroq
from langchain.prompts import ChatPromptTemplate

# Sample 5 reviews
df_test = df_cur.sample(5, random_state=155)
print("Test reviews:")
print(df_test[['review_id', 'sentiment', 'rating']])

# Zero-shot prompt
zero_prompt = ChatPromptTemplate.from_messages([
    ("system", "Classify the sentiment of the review as exactly one of: positive, negative, neutral. Return only the sentiment word."),
    ("human", "Review: {review_text}\nSentiment:")
])
llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0)
zero_chain = zero_prompt | llm

# Store results
results_zero = []
for idx, row in df_test.iterrows():
    raw_pred = zero_chain.invoke({"review_text": row['review_text']}).content.strip().lower()
    # Strip trailing punctuation (period, comma, etc.) to match ground truth exactly
    pred = raw_pred.rstrip('.,!?;')
    results_zero.append({
        "review_id": row['review_id'],
        "ground_truth": row['sentiment'],
        "zero_shot_pred": pred
    })

print("Zero-shot predictions:", results_zero)

Test reviews:
     review_id sentiment  rating
199        999  positive     4.6
169        969  negative     1.5
77         877  positive     3.9
130        930  positive     3.8
166        966   neutral     2.8
Zero-shot predictions: [{'review_id': 999, 'ground_truth': 'positive', 'zero_shot_pred': 'positive'}, {'review_id': 969, 'ground_truth': 'negative', 'zero_shot_pred': 'negative'}, {'review_id': 877, 'ground_truth': 'positive', 'zero_shot_pred': 'positive'}, {'review_id': 930, 'ground_truth': 'positive', 'zero_shot_pred': 'positive'}, {'review_id': 966, 'ground_truth': 'neutral', 'zero_shot_pred': 'neutral'}]


In [16]:
# ── T4b — Few-shot ─────────────────────────────────────────────────────────────
# Goal: Get few-shot predictions using 3 examples.
# Method: Build prompt with examples prepended.

few_shot_prompt = ChatPromptTemplate.from_messages([
    ("system", "Classify the sentiment of the review as exactly one of: positive, negative, neutral. Use the examples below."),
    ("human", "Example 1: \"Excellent work, delivered on time\" → positive"),
    ("human", "Example 2: \"Disappointed with quality, did not meet specs\" → negative"),
    ("human", "Example 3: \"Average quality, met basic requirements\" → neutral"),
    ("human", "Now classify: {review_text}\nSentiment:")
])
few_chain = few_shot_prompt | llm

# Store few-shot predictions – use enumerate to track position
for idx, (_, row) in enumerate(df_test.iterrows()):
    pred = few_chain.invoke({"review_text": row['review_text']}).content.strip().lower()
    results_zero[idx]["few_shot_pred"] = pred

print("Few-shot predictions:", [(r['review_id'], r['few_shot_pred']) for r in results_zero])

Few-shot predictions: [(999, 'positive'), (969, 'negative'), (877, 'positive'), (930, 'positive'), (966, 'neutral')]


In [17]:
# ── T4c — Comparison table and accuracy ──────────────────────────────────────
# Goal: Display results and calculate accuracies.
# Method: Loop through results, compare predictions with ground truth.

print("\nComparison Table:")
print(f"{'Review ID':>10} | {'Ground Truth':>12} | {'Zero-Shot':>12} | {'Few-Shot':>12} | {'Zero OK':>8} | {'Few OK':>8}")
print("-" * 80)

zero_correct = 0
few_correct = 0

for r in results_zero:
    z_ok = r['zero_shot_pred'] == r['ground_truth']
    f_ok = r['few_shot_pred'] == r['ground_truth']
    if z_ok: zero_correct += 1
    if f_ok: few_correct += 1
    print(f"{r['review_id']:>10} | {r['ground_truth']:>12} | {r['zero_shot_pred']:>12} | {r['few_shot_pred']:>12} | {str(z_ok):>8} | {str(f_ok):>8}")

zero_acc = zero_correct / len(results_zero)
few_acc = few_correct / len(results_zero)

print(f"\nZero-shot accuracy: {zero_acc:.0%}")
print(f"Few-shot accuracy:  {few_acc:.0%}")


Comparison Table:
 Review ID | Ground Truth |    Zero-Shot |     Few-Shot |  Zero OK |   Few OK
--------------------------------------------------------------------------------
       999 |     positive |     positive |     positive |     True |     True
       969 |     negative |     negative |     negative |     True |     True
       877 |     positive |     positive |     positive |     True |     True
       930 |     positive |     positive |     positive |     True |     True
       966 |      neutral |      neutral |      neutral |     True |     True

Zero-shot accuracy: 100%
Few-shot accuracy:  100%


### T4 NRA Insight

**Title:** *Few-Shot vs Zero-Shot Sentiment Accuracy*

- **Number:** Zero‑shot accuracy = 100%; Few‑shot accuracy = 100%.
- **Reason:** Both prompts achieve perfect accuracy after stripping trailing punctuation from the zero‑shot output; the examples in the few‑shot prompt disambiguate the format, but the zero‑shot model is also capable when its output is cleaned.
- **Action:** We will adopt the few‑shot prompt as the production standard for its robustness, but we will also implement a string‑cleaning post‑processor for all sentiment outputs (strip punctuation, whitespace) to guarantee 100% matching against ground‑truth labels.

---
# TASK 5 — Integrated NRA Business Report (20 pts)

**Goal:** Combine findings from Tasks 1–4 into a 3-bullet executive NRA report.

| Sub-task | Points |
|----------|--------|
| T5a — 3 NRA bullets, each with number from prior printed output | 20 |

**Requirements:**
- Each bullet covers a different task signal (T1 ML, T2 risk, T3 drift or T4 prompting)
- Each Number is a **single anchor stat** from a prior task's printed output
- Reason must state a causal mechanism (not describe the outcome)
- Action must be specific and committed (concrete parameters/steps named)
- Bullets must be **internally consistent** — no contradictory recommendations

**Bullet structure:**
```
## 🔷 ReviewPulse Intelligence — Executive NRA Report

**Bullet 1 — Classifier Performance (from T1)**
Number: ___
Reason: ___
Action: ___

**Bullet 2 — High-Risk Review Volume (from T2)**
Number: ___
Reason: ___
Action: ___

**Bullet 3 — Monitoring Status (from T3 or T4)**
Number: ___
Reason: ___
Action: ___
```

## 🔷 ReviewPulse Intelligence — Executive NRA Report

**Bullet 1 — Classifier Performance (from T1)**
- **Number:** Best ROC‑AUC = 0.833.
- **Reason:** The rating feature is a strong proxy for re‑hire probability, enabling a simple logistic model to achieve robust discrimination.
- **Action:** Deploy `HiredAgainClassifier` v1 as a production microservice; set a monitoring dashboard in MLflow to track daily prediction drift and retrain automatically if accuracy drops below 0.80 for three consecutive days.

**Bullet 2 — High-Risk Review Volume (from T2)**
- **Number:** 45 urgent cases identified in the current window.
- **Reason:** All reviews with rating < 2.0 contain critical failure signals (e.g., “missed deadlines,” “poor quality”), necessitating immediate intervention to retain clients.
- **Action:** Implement a priority queue for urgent reviews; assign a dedicated support agent to each case within 1 hour of detection, and escalate high‑risk cases to the platform’s trust & safety team within 24 hours.

**Bullet 3 — Monitoring Status (from T3)**
- **Number:** 0.00% drifted columns.
- **Reason:** Both reference and current windows are sampled from the same underlying review‑generation process; no meaningful shift in rating or re‑hire distributions has occurred, so the distributions are statistically indistinguishable (0% drift).
- **Action:** Run a weekly Evidently pipeline; if drift exceeds 0.4 for two consecutive weeks, trigger a model retraining pipeline and alert the data science team via Slack. Maintain this cycle to ensure long‑term stability.

---
# ★ BONUS — LangChain Agent with Dataset Tool (10★)

**Goal:** Build a LangChain agent with a custom tool that queries the ReviewPulse dataset
and answers the question: *"How many reviews in the current window have rating below 2.0 and what is their average rating?"*

| Sub-task | Points |
|----------|--------|
| ★a — Define custom tool `query_low_rating_reviews` that filters df_cur and returns count + avg_rating | 5★ |
| ★b — Build agent, invoke with the question above, print agent output | 5★ |

**Tool hint:**
```python
from langchain.tools import tool

@tool
def query_low_rating_reviews(threshold: float) -> str:
    """Returns count and average rating of current reviews below a given rating threshold."""
    # YOUR CODE HERE
```

**Agent hint:** Use `initialize_agent` with `AgentType.ZERO_SHOT_REACT_DESCRIPTION`

In [19]:
# ── ★ BONUS — LangChain Agent ──────────────────────────────────────────────────
# Goal: Build an agent that answers questions about low‑rating reviews.
# Method: Define a custom tool that queries df_cur, then use initialize_agent.

from langchain.agents import initialize_agent, AgentType
from langchain.tools import tool
from langchain_groq import ChatGroq

@tool
def query_low_rating_reviews(threshold: str) -> str:
    """
    Returns count and average rating of current reviews below a given rating threshold.
    Input must be a number (e.g., "2.0").
    """
    try:
        thresh = float(threshold)
    except ValueError:
        return "Error: threshold must be a number."
    filtered = df_cur[df_cur['rating'] < thresh]
    count = len(filtered)
    avg = filtered['rating'].mean() if count > 0 else 0.0
    return f"Count: {count}, Average rating: {avg:.2f}"

# Create LLM and agent
llm_agent = ChatGroq(model="llama-3.1-8b-instant", temperature=0)
tools = [query_low_rating_reviews]
agent = initialize_agent(
    tools, llm_agent,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True,
    max_iterations=3          # ← prevents infinite loop
)

# Invoke with question
question = "How many reviews in the current window have rating below 2.0 and what is their average rating?"
response = agent.invoke({"input": question})
print("\nAgent response:")
print(response['output'])



> Entering new AgentExecutor chain...
Thought: To find the number of reviews and their average rating below a certain threshold, I need to use the query_low_rating_reviews function. I should pass the threshold as a string, as specified in the function parameters.

Action: query_low_rating_reviews
Action Input: "2.0"
Observation: Count: 45, Average rating: 1.44
Thought:Question: How many reviews in the current window have rating below 2.0 and what is their average rating?
Thought: To find the number of reviews and their average rating below a certain threshold, I need to use the query_low_rating_reviews function. I should pass the threshold as a string, as specified in the function parameters.

Action: query_low_rating_reviews
Action Input: "2.0"
Observation: Count: 45, Average rating: 1.44
Thought:Question: How many reviews in the current window have rating below 2.0 and what is their average rating?
Thought: To find the number of reviews and their average rating below a certain thre

---
# SCORING RUBRIC

## Point Breakdown

| Task | Sub-task | Max | Deduction triggers |
|------|----------|-----|--------------------|
| **T1** | T1a — encode + split correct | 4 | −2 missing stratify; −2 wrong test_size |
| | T1b — 3 runs logged with all 3 metrics + param C | 8 | −2 per missing metric/param per run |
| | T1c — best run identified + model registered | 5 | −3 wrong model name; −2 no registry call |
| | T1d — NRA (number from output) | 3 | −3 number not from output; −1 weak action |
| **T2** | T2a — filter rating<2.0, correct Document count | 4 | −4 wrong filter threshold or count |
| | T2b — Pydantic schema + LCEL chain correct | 8 | −4 missing format_instructions; −4 schema wrong |
| | T2c — loop runs, total_urgent printed, NRA | 8 | −3 no loop; −2 NRA number not from output |
| **T3** | T3a — hired_again as int, shapes correct | 4 | −2 wrong encoding; −2 wrong columns |
| | T3b — ColumnDriftMetric both cols, drift+pval printed | 8 | −3 per missing column metric |
| | T3c — DataDriftPreset, share printed, NRA | 8 | −4 wrong path to share_of_drifted_columns |
| **T4** | T4a — 5 reviews sampled correctly, zero-shot run | 7 | −3 wrong random_state; −2 preds not stripped/lowered |
| | T4b — few-shot with 3 examples, preds stored | 7 | −3 no examples in prompt; −2 same prompt as T4a |
| | T4c — table printed, both accuracies computed, NRA | 6 | −3 accuracy not computed; −2 NRA not from output |
| **T5** | 3-bullet NRA: numbers from output, no contradictions | 20 | −5 per bullet with number not from output; −5 contradicting bullets |
| **★** | Tool correct structure + output format | 5★ | |
| | Agent invokes tool, returns coherent answer | 5★ | |
| **TOTAL** | | **100 + 10★** | |

---

## Automatic Full Deductions (−20% per violation)
| Violation | Deduction |
|-----------|----------|
| Raw data cell modified | −20 pts |
| Any NRA Number typed from memory (not from printed output) | −5 pts per bullet |
| NRA Reason describes outcome instead of mechanism | −2 pts per bullet |
| NRA Action uses hedging language ("consider", "might", "explore") | −1 pt per bullet |
| Contradicting recommendations across T5 bullets | −5 pts |
| `savefig()` after `plt.show()` where plot is saved | −2 pts |

---
# INTERVIEW ANSWER

**Q: You've worked with LangChain, MLflow, and Evidently — how do they fit together in a production AI system?**

*"These three tools cover different layers of a production AI pipeline, and I've used all three on the same dataset in this capstone.*

*MLflow handles the ML lifecycle — I log experiment runs with different hyperparameters, compare metrics, and register the best model in the Model Registry. That gives every model a version number and a deployment-ready URI.*

*LangChain handles the LLM layer. On top of the ML model, I build LCEL chains with PydanticOutputParser so the LLM's response is always a typed Python object — not raw text. This is critical in production because downstream code needs guaranteed field types, not free-form strings.*

*Evidently handles monitoring. After the model is deployed, I compare the reference distribution (training data) against live data using ColumnDriftMetric. If `share_of_drifted_columns` crosses a threshold, that's an automated alert to retrain.*

*Together: MLflow builds and tracks, LangChain orchestrates and structures, Evidently monitors. A client gets a system that doesn't just work on launch day — it tells you when it's starting to fail."*

---

**GitHub commit when done:**
```
feat: Day180 - Month10 Capstone ReviewPulse Intelligence Platform [score/100+bonus★]
```